# SQL Analysis
#### Extraction of data from the cleaned datasets using sqlite
**Input:** Cleaned CSV filed for the cleaned_data folder  
**Author: Kunal Verma**

In [2]:
import pandas as pd
import sqlite3

In [3]:
df_h1b = pd.read_csv('../data/cleaned_data/h1b_cleaned.csv')
df_ppp = pd.read_csv('../data/cleaned_data/ppp_cleaned.csv')
df_levels = pd.read_csv('../data/cleaned_data/levels_cleaned.csv')

print('H1B shape: ',df_h1b.shape)
print('Levels shape: ',df_levels.shape)
print('PPP shape: ',df_ppp.shape)

H1B shape:  (189626, 6)
Levels shape:  (15273, 6)
PPP shape:  (189, 3)


In [4]:
conn = sqlite3.connect('../data/salary_benchmark.db')
df_h1b.to_sql('h1b', conn, if_exists = 'replace', index = False)
df_levels.to_sql('levels', conn, if_exists = 'replace', index = False)
df_ppp.to_sql('ppp', conn, if_exists = 'replace', index = False)

print("Database has been created and the tables have been loaded successfully.")

Database has been created and the tables have been loaded successfully.


In [16]:
#1. Avg salary by country

Q1 = pd.read_sql_query(""" Select employee_residence as country, Round(Avg(salary_in_usd), 0) as avg_salary,
Round(Min(salary_in_usd), 0) as min_salary, Round(Max(salary_in_usd), 0) as max_salary, count(*) as total_records
from levels group by country order by avg_salary DESC """, conn)
print('The first Query: To get the Average Salary Countrywise \n')
print(Q1)

The first Query: To get the Average Salary Countrywise 

  country  avg_salary  min_salary  max_salary  total_records
0      US    157340.0     24000.0    750000.0          14388
1      DE    101518.0     24823.0    275000.0             94
2      GB     93608.0     28299.0    430967.0            685
3      NL     76479.0     32390.0    134960.0             31
4      IN     47494.0     15809.0    200000.0             75


In [18]:
#2. Salary by experience level per Country 

Q2 = pd.read_sql_query(''' select employee_residence as country, experience_level, round(avg(salary_in_usd),0) as avg_salary, count(*) as
total_records from levels group by employee_residence, experience_level order by country, avg_salary desc ''', conn)
print('The Second query: To get the salary by experience in each country \n')
print(Q2)

The Second query: To get the salary by experience in each country 

   country experience_level  avg_salary  total_records
0       DE               EX    135936.0              2
1       DE               SE    127963.0             42
2       DE               MI     83549.0             31
3       DE               EN     68756.0             19
4       GB               EX    158722.0             22
5       GB               SE    113809.0            259
6       GB               MI     83483.0            314
7       GB               EN     54883.0             90
8       IN               EX    113155.0              2
9       IN               SE     71317.0             22
10      IN               MI     35191.0             30
11      IN               EN     33860.0             21
12      NL               EX    107887.0              2
13      NL               SE     95257.0              9
14      NL               MI     78669.0              9
15      NL               EN     53613.0             

In [31]:
#3. Top highest paid tech job roles in the US
Q3 = pd.read_sql_query('''select job_title, round(avg(salary_in_usd),0) as avg_salary, count(*) as total_records from levels where 
employee_residence = 'US'
group by job_title 
having count(*) > 1 
order by avg_salary desc 
limit 10''', conn)
print('The Third Query: Top 10 highest paid tech job roles in the US \n')
print(Q3)

The Third Query: Top 10 highest paid tech job roles in the US 

                            job_title  avg_salary  total_records
0            Head of Machine Learning    337000.0              6
1      Managing Director Data Science    280000.0              2
2            Director of Data Science    244115.0             22
3              Applied Data Scientist    238000.0              3
4                        AI Architect    235694.0             26
5                        Head of Data    232892.0             48
6                Head of Data Science    224539.0              7
7              Deep Learning Engineer    218855.0              8
8            Computer Vision Engineer    217686.0             21
9  Machine Learning Software Engineer    215622.0              9


In [34]:
#4. Salary gap in every job title: Ind vs US
Q4 = pd.read_sql_query('''select us.job_title, round(ind.avg_salary, 0) as india_avg, round(us.avg_salary, 0) as us_avg,
round((us.avg_salary - ind.avg_salary)/us.avg_salary * 100,1) as gap_pjt 
from (select job_title, avg(salary_in_usd) as avg_salary from levels where employee_residence = 'US'
                       group by job_title having count(*) > 2) us
                       JOIN
                       (select job_title, avg(salary_in_usd) as avg_salary from levels where employee_residence = "IN"
                       group by job_title having count(*) > 1) ind
                       ON
                       us.job_title = ind.job_title
                       order by gap_pjt DESC limit 10 ''', conn)
print("The Fourth Query: IND vs US salary gap in every job title")
print(Q4)

The Fourth Query: IND vs US salary gap in every job title
                   job_title  india_avg    us_avg  gap_pjt
0               Data Analyst    20847.0  112165.0     81.4
1             Data Scientist    33994.0  163142.0     79.2
2  Machine Learning Engineer    43599.0  195968.0     77.8
3          Lead Data Analyst    25822.0  108333.0     76.2
4     Data Analytics Manager    32958.0  138081.0     76.1
5   Computer Vision Engineer    52949.0  217686.0     75.7
6      Business Data Analyst    28706.0  109444.0     73.8
7        Lead Data Scientist    48647.0  179623.0     72.9
8       Data Science Manager    57863.0  202160.0     71.4
9              Data Engineer    54043.0  153231.0     64.7


In [37]:
#5. H1B average salary by job title

Q5 = pd.read_sql_query('''select job_title, round(avg(wage_rate_of_pay_from), 0) as avg_salary, count(*) as total_records
from h1b group by job_title having count(*) > 50 order by avg_salary DESC limit 15''', conn)
print('The Fifth query: Average H1B salary by job titles (Top 15 highest paid jobs.) \n')
print(Q5)

The Fifth query: Average H1B salary by job titles (Top 15 highest paid jobs.) 

                          job_title  avg_salary  total_records
0             Hospitalist Physician    295653.0            192
1                      Psychiatrist    292860.0             56
2                      Nephrologist    289574.0             76
3                         Physician    288074.0            189
4                       Hospitalist    284473.0            255
5                 Managing Director    274864.0             53
6         Family Medicine Physician    247832.0             83
7                          Attorney    243979.0             66
8       Internal Medicine Physician    242181.0             64
9         Software Engineering LMTS    237004.0            105
10          Chief Executive Officer    235838.0             87
11  Staff Machine Learning Engineer    235264.0             58
12     Director, Product Management    229914.0             59
13                        Principal   

In [39]:
#6. PPP factor for the 5 countries we're analysing

Q6 = pd.read_sql_query('''select country, country_code, ppp_2024 from ppp where country_code IN ('IND', 'USA', 'GBR', 'DEU', 'NLD')
order by ppp_2024''', conn)
print('The Sixth Query: PPP factor for the 5 countries we are analysing \n')
print(Q6)

The Sixth Query: PPP factor for the 5 countries we are analysing 

          country country_code   ppp_2024
0  United Kingdom          GBR   0.664153
1         Germany          DEU   0.700862
2     Netherlands          NLD   0.731421
3   United States          USA   1.000000
4           India          IND  20.421988


In [41]:
conn.close()
print('Connection to the database has closed.')

Connection to the database has closed.
